
# OLMo-2 RI / Semantic Induction Head Developmental Analysis

This notebook analyzes the **completed RI developmental sweep only**. It is intentionally separate from the expensive sweep notebook and never loads an OLMo checkpoint.

The sweep covers:

- 10 Stage-1 checkpoints;
- 7 AGENDA semantic relations;
- 100 frozen tokenizer-safe triplets per relation;
- all 32 × 32 = 1024 attention heads;
- the relaxed developmental QK gate (`source == attention argmax`) as the primary definition;
- the original `tau = 2.2` gate as a sensitivity analysis;
- random-token and token-10 null controls;
- fixed final-checkpoint high-RI heads traced backward.

The central analysis question is:

> **Does semantic-relation structure change around the same early training interval in which the behavioral ICL sweep changes (roughly 3B–5B tokens / steps 600–1000)?**

A second question is whether the **mature high-RI head population** already exists in that early interval or forms/reorganizes later.

### Important metric distinction

The saved sweep contains three conceptually different quantities:

\[
\text{conditional RI} = \texttt{ri\_argmax},
\]

which measures target-token promotion **conditional on the head routing to the source**;

\[
\text{QK source-routing frequency} = \texttt{qk\_argmax\_frequency},
\]

which measures how often the head's attention argmax is the source; and

\[
\text{RI per opportunity} = \texttt{ri\_per\_opportunity},
\]

which combines the two. These must not be conflated. A developmental increase in routing frequency is not automatically an increase in conditional OV relation selectivity.


# 1. Setup


In [ ]:

from google.colab import drive
drive.mount("/content/drive")


In [ ]:

from pathlib import Path
import json
import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/NLP_Project/olmo_sih_dynamics")

RI_RUN_DIR = ROOT / "results" / "ri" / "developmental_v1"
RI_ANALYSIS_DIR = RI_RUN_DIR / "analysis"
FIG_DIR = RI_ANALYSIS_DIR / "figures"

ICL_RESULTS_DIR = ROOT / "results" / "icl"
ICL_ANALYSIS_DIR = ICL_RESULTS_DIR / "analysis"

RI_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("RI results:", RI_RUN_DIR)
print("RI analysis output:", RI_ANALYSIS_DIR)


# 2. Load Saved RI Sweep Outputs


In [ ]:

required_files = {
    "config": RI_RUN_DIR / "run_config.json",
    "checkpoints": RI_RUN_DIR / "selected_checkpoint_manifest.csv",
    "trajectory": RI_RUN_DIR / "ri_head_trajectory.csv",
    "checkpoint_summary": RI_RUN_DIR / "ri_checkpoint_summary.csv",
    "final_top_heads": RI_RUN_DIR / "final_top_heads.csv",
    "final_trace": RI_RUN_DIR / "final_top_heads_backward_trace.csv",
    "stability": RI_RUN_DIR / "head_stability_vs_final.csv",
    "assessment": RI_RUN_DIR / "ri_assessment_manifest.jsonl",
    "excluded": RI_RUN_DIR / "ri_excluded_self_relations.jsonl",
}

missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing RI result files:\n" + "\n".join(missing)
    )

run_config = json.loads(required_files["config"].read_text())
checkpoint_manifest = pd.read_csv(required_files["checkpoints"])
ri = pd.read_csv(required_files["trajectory"])
checkpoint_summary = pd.read_csv(required_files["checkpoint_summary"])
final_top_heads = pd.read_csv(required_files["final_top_heads"])
final_trace = pd.read_csv(required_files["final_trace"])
stability = pd.read_csv(required_files["stability"])

assessment = [
    json.loads(line)
    for line in required_files["assessment"].read_text().splitlines()
    if line.strip()
]
excluded = [
    json.loads(line)
    for line in required_files["excluded"].read_text().splitlines()
    if line.strip()
]

print("Run configuration:")
print(json.dumps(run_config, indent=2))


# 3. Integrity and Completeness Audit


In [ ]:

EXPECTED_RELATIONS = sorted(run_config["relations"])
EXPECTED_STEPS = list(run_config["target_steps"])
EXPECTED_HEADS = 32 * 32
EXPECTED_ROWS = len(EXPECTED_STEPS) * len(EXPECTED_RELATIONS) * EXPECTED_HEADS

print("Trajectory rows:", len(ri), "/", EXPECTED_ROWS)
print("Checkpoints:", ri["step"].nunique(), "/", len(EXPECTED_STEPS))
print("Relations:", ri["relation"].nunique(), "/", len(EXPECTED_RELATIONS))
print("Directions:", sorted(ri["direction"].unique()))
print("Assessment triplets:", len(assessment))
print("Excluded self-relations:", len(excluded))

assert len(ri) == EXPECTED_ROWS
assert sorted(ri["step"].unique()) == sorted(EXPECTED_STEPS)
assert sorted(ri["relation"].unique()) == EXPECTED_RELATIONS
assert set(ri["direction"]) == {"forward"}

coverage = (
    ri.groupby(["step", "relation"])
    .size()
    .rename("n_heads")
    .reset_index()
)
assert (coverage["n_heads"] == EXPECTED_HEADS).all()

assessment_counts = pd.Series(
    Counter(row["relation"] for row in assessment),
    name="n",
).sort_index()

print("\nFrozen assessment rows per relation:")
print(assessment_counts.to_string())

assert set(assessment_counts.index) == set(EXPECTED_RELATIONS)
assert assessment_counts.nunique() == 1
assert assessment_counts.iloc[0] == run_config["rows_per_relation"]

print("\nPASS: RI sweep is complete and balanced.")


## 3.1 Checkpoint labels


In [ ]:

checkpoint_manifest = checkpoint_manifest.sort_values("step").reset_index(drop=True)

checkpoint_manifest["label"] = checkpoint_manifest.apply(
    lambda r: f"{int(r.tokens_B)}B\nstep {int(r.step)}",
    axis=1,
)
checkpoint_manifest["index"] = np.arange(len(checkpoint_manifest))

STEP_ORDER = checkpoint_manifest["step"].tolist()
STEP_TO_X = dict(zip(checkpoint_manifest["step"], checkpoint_manifest["index"]))
STEP_TO_LABEL = dict(zip(checkpoint_manifest["step"], checkpoint_manifest["label"]))

display(checkpoint_manifest)



# 4. Current-Run Executive Summary

The following interpretation is deliberately conservative and separates **routing** from **conditional RI**.

From the completed sweep:

- mean source-routing frequency rises from about **0.053 at 1B / step 150** to a maximum of about **0.080 at 4B / step 850**, then declines;
- the strict \(\tau=2.2\) source-routing frequency rises from about **0.0037** to about **0.0216** over the same early interval;
- mean RI-per-opportunity likewise peaks near **4B / step 850**;
- in contrast, mean **conditional RI given source routing** stays roughly flat around 0.08 throughout the 1B–13B early regime and rises more clearly only later;
- supported per-checkpoint top-15 conditional RI is also fairly flat through the 3B–13B region, then grows at 38B and especially at the final 3896B checkpoint;
- the identity of the final mature top-RI heads is **not stable in the early window**: top-k Jaccard overlap with the final set is near zero at 3B–5B, while all-head rank similarity grows only gradually.

So the strongest early signal in this run is currently:

\[
\boxed{\text{a transient increase in source-directed QK routing around 3B–5B}}
\]

rather than a population-wide jump in conditional OV Relation Index itself.

That timing overlaps the independently measured behavioral ICL transition, but it is **not yet evidence that RI/SIH formation causes ICL**. The rest of this notebook tests this statement relation-by-relation, with support thresholds, null controls, strict-gate sensitivity, head-set stability, and direct comparison to the ICL trajectory.


# 5. Population-Level Developmental Trajectory


In [ ]:

# Exact source-gated frequency that has a valid RI denominator.
ri["qk_scored_frequency"] = (
    ri["argmax_scored"] / ri["opportunities"]
)

population = (
    ri.groupby(["step", "tokens_B"])
    .agg(
        mean_ri_argmax=("ri_argmax", "mean"),
        median_ri_argmax=("ri_argmax", "median"),
        mean_ri_per_opportunity=("ri_per_opportunity", "mean"),
        mean_qk_argmax_frequency=("qk_argmax_frequency", "mean"),
        mean_qk_scored_frequency=("qk_scored_frequency", "mean"),
        mean_qk_strict_frequency=("qk_strict_frequency", "mean"),
        mean_ri_strict=("ri_strict_tau_2_2", "mean"),
        mean_null_random=("null_random_mean", "mean"),
        mean_ri_minus_null_random=("ri_minus_null_random", "mean"),
        mean_ri_minus_null_tok10=("ri_minus_null_tok10", "mean"),
    )
    .reset_index()
    .sort_values("step")
)

top15_balanced = (
    checkpoint_summary.groupby(["step", "tokens_B"])
    .agg(
        mean_supported_top15_ri=("top15_ri_mean", "mean"),
        mean_summary_qk=("mean_qk_argmax_frequency", "mean"),
        mean_summary_per_opportunity=("mean_ri_per_opportunity", "mean"),
    )
    .reset_index()
)

population = population.merge(
    top15_balanced,
    on=["step", "tokens_B"],
    how="left",
)

population.to_csv(
    RI_ANALYSIS_DIR / "ri_population_trajectory.csv",
    index=False,
)

display(population.round(6))


In [ ]:

def categorical_x(df):
    return df["step"].map(STEP_TO_X).to_numpy()

def apply_checkpoint_axis(ax):
    ax.set_xticks(checkpoint_manifest["index"])
    ax.set_xticklabels(checkpoint_manifest["label"], rotation=45, ha="right")
    ax.set_xlabel("Training checkpoint (categorical spacing)")
    ax.grid(alpha=0.25)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    categorical_x(population),
    population["mean_qk_argmax_frequency"],
    marker="o",
    label="argmax source-routing frequency",
)
ax.plot(
    categorical_x(population),
    population["mean_qk_strict_frequency"],
    marker="o",
    label="strict tau=2.2 routing frequency",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Mean frequency across relation × head cells")
ax.set_title("Source-directed QK routing over training")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "qk_routing_population.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    categorical_x(population),
    population["mean_ri_argmax"],
    marker="o",
    label="mean conditional RI",
)
ax.plot(
    categorical_x(population),
    population["mean_supported_top15_ri"],
    marker="o",
    label="mean supported top-15 RI",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Relation Index")
ax.set_title("Conditional Relation Index over training")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "conditional_ri_population.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    categorical_x(population),
    population["mean_ri_per_opportunity"],
    marker="o",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Mean RI per source-routing opportunity")
ax.set_title("Combined RI-per-opportunity signal")
fig.tight_layout()
fig.savefig(FIG_DIR / "ri_per_opportunity_population.png", dpi=180, bbox_inches="tight")
plt.show()


# 6. Zoom In on the 3B–5B Behavioral Transition Window


In [ ]:

EARLY_STEPS = [600, 700, 850, 900, 1000]

early_population = population[
    population["step"].isin(EARLY_STEPS)
].copy()

display(
    early_population[
        [
            "step",
            "tokens_B",
            "mean_qk_argmax_frequency",
            "mean_qk_strict_frequency",
            "mean_ri_argmax",
            "mean_ri_per_opportunity",
            "mean_supported_top15_ri",
        ]
    ].round(6)
)


In [ ]:

early_relation = checkpoint_summary[
    checkpoint_summary["step"].isin(EARLY_STEPS)
][
    [
        "step",
        "tokens_B",
        "relation",
        "top15_ri_mean",
        "mean_ri_per_opportunity",
        "mean_qk_argmax_frequency",
    ]
].copy()

baseline = (
    early_relation[early_relation["step"] == 600]
    .set_index("relation")
)

change_rows = []

for row in early_relation.itertuples(index=False):
    b = baseline.loc[row.relation]
    change_rows.append({
        "step": row.step,
        "tokens_B": row.tokens_B,
        "relation": row.relation,
        "delta_qk_vs_step600": (
            row.mean_qk_argmax_frequency - b.mean_qk_argmax_frequency
        ),
        "delta_per_opportunity_vs_step600": (
            row.mean_ri_per_opportunity - b.mean_ri_per_opportunity
        ),
        "delta_top15_conditional_ri_vs_step600": (
            row.top15_ri_mean - b.top15_ri_mean
        ),
    })

early_changes = pd.DataFrame(change_rows)
early_changes.to_csv(
    RI_ANALYSIS_DIR / "ri_early_window_relation_changes.csv",
    index=False,
)

print("Changes relative to 3B / step 600:")
display(
    early_changes[
        early_changes["step"].isin([850, 900, 1000])
    ].round(6)
)


In [ ]:

fig, ax = plt.subplots(figsize=(11, 6))

for relation, group in early_relation.groupby("relation"):
    group = group.sort_values("step")
    ax.plot(
        group["step"].map(STEP_TO_X),
        group["mean_qk_argmax_frequency"],
        marker="o",
        label=relation,
    )

ax.set_xticks([STEP_TO_X[s] for s in EARLY_STEPS])
ax.set_xticklabels(
    [STEP_TO_LABEL[s] for s in EARLY_STEPS],
    rotation=45,
    ha="right",
)
ax.set_xlabel("Training checkpoint")
ax.set_ylabel("Mean source-routing frequency")
ax.set_title("Relation-specific QK routing in the 3B–5B window")
ax.grid(alpha=0.25)
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "qk_routing_early_by_relation.png", dpi=180, bbox_inches="tight")
plt.show()



### How to read the early-window result

A useful test is whether the same pattern is visible across relations rather than being driven by one category.

For this run, the strongest early-window change is in **source-routing frequency**. Several relations increase markedly between step 600 and the 4B checkpoints, while conditional top-15 RI does not show a common upward jump across all seven relations.

This distinction matters: the data are compatible with an early change in **where heads route attention**, without requiring a simultaneous global increase in **how strongly the raw OV circuit favors the semantic tail once that routing event has already occurred**.


# 7. Exact Decomposition: Routing × Conditional RI


In [ ]:

# Rowwise, this identity should hold:
#
#   ri_per_opportunity
# = (argmax_scored / opportunities) * ri_argmax
#
# except where ri_argmax is NaN because there were no scored events.

expected = ri["qk_scored_frequency"] * ri["ri_argmax"]
mask = expected.notna() & ri["ri_per_opportunity"].notna()

max_error = np.max(
    np.abs(
        expected[mask].to_numpy()
        - ri.loc[mask, "ri_per_opportunity"].to_numpy()
    )
)

print("Maximum absolute rowwise decomposition error:", max_error)
assert max_error < 1e-12

print("PASS: RI per opportunity decomposes exactly into routing frequency × conditional RI.")


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    categorical_x(population),
    population["mean_qk_scored_frequency"],
    marker="o",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Mean scored source-routing frequency")
ax.set_title("Routing component of RI-per-opportunity")
fig.tight_layout()
fig.savefig(FIG_DIR / "ri_decomposition_routing_component.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    categorical_x(population),
    population["mean_ri_argmax"],
    marker="o",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Mean conditional RI")
ax.set_title("Conditional-OV component of RI-per-opportunity")
fig.tight_layout()
fig.savefig(FIG_DIR / "ri_decomposition_conditional_component.png", dpi=180, bbox_inches="tight")
plt.show()



The decomposition prevents an important interpretive error. If `ri_per_opportunity` rises while conditional `ri_argmax` remains flat, the developmental change is mainly due to heads **routing to the relation source more often**, rather than becoming more selective for the tail *conditional on already routing to the source*.


# 8. Relation-Specific Developmental Heat Tables


In [ ]:

qk_table = (
    checkpoint_summary.pivot(
        index="relation",
        columns="step",
        values="mean_qk_argmax_frequency",
    )
    .reindex(index=EXPECTED_RELATIONS, columns=STEP_ORDER)
)

top15_table = (
    checkpoint_summary.pivot(
        index="relation",
        columns="step",
        values="top15_ri_mean",
    )
    .reindex(index=EXPECTED_RELATIONS, columns=STEP_ORDER)
)

peropp_table = (
    checkpoint_summary.pivot(
        index="relation",
        columns="step",
        values="mean_ri_per_opportunity",
    )
    .reindex(index=EXPECTED_RELATIONS, columns=STEP_ORDER)
)

print("Mean QK argmax frequency")
display(qk_table.round(4))

print("Supported top-15 conditional RI")
display(top15_table.round(4))

print("Mean RI per opportunity")
display(peropp_table.round(5))


In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(qk_table.to_numpy(), aspect="auto")
ax.set_yticks(np.arange(len(qk_table.index)))
ax.set_yticklabels(qk_table.index)
ax.set_xticks(np.arange(len(STEP_ORDER)))
ax.set_xticklabels(
    [STEP_TO_LABEL[s] for s in STEP_ORDER],
    rotation=45,
    ha="right",
)
ax.set_title("Mean source-routing frequency by semantic relation")
ax.set_xlabel("Checkpoint")
fig.colorbar(im, ax=ax, label="QK argmax frequency")
fig.tight_layout()
fig.savefig(FIG_DIR / "qk_relation_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(top15_table.to_numpy(), aspect="auto")
ax.set_yticks(np.arange(len(top15_table.index)))
ax.set_yticklabels(top15_table.index)
ax.set_xticks(np.arange(len(STEP_ORDER)))
ax.set_xticklabels(
    [STEP_TO_LABEL[s] for s in STEP_ORDER],
    rotation=45,
    ha="right",
)
ax.set_title("Supported top-15 conditional RI by semantic relation")
ax.set_xlabel("Checkpoint")
fig.colorbar(im, ax=ax, label="Top-15 conditional RI")
fig.tight_layout()
fig.savefig(FIG_DIR / "top15_ri_relation_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()


# 9. Strict τ=2.2 Sensitivity


In [ ]:

strict_summary = (
    ri.groupby(["step", "tokens_B"])
    .agg(
        relaxed_qk_frequency=("qk_argmax_frequency", "mean"),
        strict_qk_frequency=("qk_strict_frequency", "mean"),
        relaxed_conditional_ri=("ri_argmax", "mean"),
        strict_conditional_ri=("ri_strict_tau_2_2", "mean"),
    )
    .reset_index()
    .sort_values("step")
)

strict_summary.to_csv(
    RI_ANALYSIS_DIR / "ri_strict_gate_sensitivity.csv",
    index=False,
)

display(strict_summary.round(6))


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    strict_summary["step"].map(STEP_TO_X),
    strict_summary["relaxed_qk_frequency"],
    marker="o",
    label="relaxed argmax gate",
)
ax.plot(
    strict_summary["step"].map(STEP_TO_X),
    strict_summary["strict_qk_frequency"],
    marker="o",
    label="strict tau=2.2 gate",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Mean source-routing frequency")
ax.set_title("Relaxed vs. strict QK gate")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "strict_gate_frequency.png", dpi=180, bbox_inches="tight")
plt.show()



The strict gate is not the primary developmental metric, because early checkpoints may not yet contain sharply concentrated attention. It is nevertheless useful as a sensitivity check: if both relaxed and strict source-routing frequencies change in the same early interval, the result is less likely to be caused only by a diffuse-attention artifact.


# 10. Null Controls


In [ ]:

# Recompute supported top-15 values together with their null controls.
top15_null_rows = []

for (step, tokens_B, relation), group in ri.groupby(
    ["step", "tokens_B", "relation"]
):
    eligible = group[group["argmax_scored"] >= 10].copy()
    top = eligible.nlargest(15, "ri_argmax")

    top15_null_rows.append({
        "step": step,
        "tokens_B": tokens_B,
        "relation": relation,
        "n_top": len(top),
        "top15_ri": top["ri_argmax"].mean(),
        "top15_random_null": top["null_random_mean"].mean(),
        "top15_ri_minus_random": top["ri_minus_null_random"].mean(),
        "top15_ri_minus_tok10": top["ri_minus_null_tok10"].mean(),
    })

top15_null = pd.DataFrame(top15_null_rows)
top15_null.to_csv(
    RI_ANALYSIS_DIR / "ri_top15_null_controls.csv",
    index=False,
)

null_population = (
    top15_null.groupby(["step", "tokens_B"])
    .mean(numeric_only=True)
    .reset_index()
    .sort_values("step")
)

display(null_population.round(6))


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    null_population["step"].map(STEP_TO_X),
    null_population["top15_ri"],
    marker="o",
    label="supported top-15 target RI",
)
ax.plot(
    null_population["step"].map(STEP_TO_X),
    null_population["top15_random_null"],
    marker="o",
    label="same heads: random visible-token null",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Conditional score")
ax.set_title("Supported top-RI heads versus random-token null")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "top15_random_null.png", dpi=180, bbox_inches="tight")
plt.show()



Because heads are selected using their target RI, the target-minus-null gap for the selected top set is **descriptive and selection-biased**. It is useful as a sanity check, not as an independent significance test.


# 11. Mature Top-RI Heads Traced Backward


In [ ]:

fixed_final_summary = (
    final_trace.groupby(["step", "tokens_B"])
    .agg(
        fixed_final_top_ri=("ri_argmax", "mean"),
        fixed_final_top_per_opportunity=("ri_per_opportunity", "mean"),
        fixed_final_top_qk=("qk_argmax_frequency", "mean"),
        fixed_final_top_random_gap=("ri_minus_null_random", "mean"),
        fixed_final_top_strict_ri=("ri_strict_tau_2_2", "mean"),
    )
    .reset_index()
    .sort_values("step")
)

fixed_final_summary.to_csv(
    RI_ANALYSIS_DIR / "fixed_final_top_heads_trajectory.csv",
    index=False,
)

display(fixed_final_summary.round(6))


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    fixed_final_summary["step"].map(STEP_TO_X),
    fixed_final_summary["fixed_final_top_ri"],
    marker="o",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Mean conditional RI")
ax.set_title("Final-checkpoint top-RI heads traced backward at fixed coordinates")
fig.tight_layout()
fig.savefig(FIG_DIR / "final_top_heads_backward_ri.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:

stability_population = (
    stability.groupby(["step", "tokens_B"])
    .agg(
        mean_topk_jaccard_vs_final=("topk_jaccard_vs_final", "mean"),
        mean_all_head_spearman_vs_final=("all_head_spearman_vs_final", "mean"),
        mean_eligible_heads=("eligible_heads", "mean"),
    )
    .reset_index()
    .sort_values("step")
)

stability_population.to_csv(
    RI_ANALYSIS_DIR / "ri_stability_population.csv",
    index=False,
)

display(stability_population.round(5))


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    stability_population["step"].map(STEP_TO_X),
    stability_population["mean_topk_jaccard_vs_final"],
    marker="o",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Mean top-15 Jaccard vs final")
ax.set_ylim(-0.02, 1.05)
ax.set_title("Identity stability of high-RI heads relative to the final checkpoint")
fig.tight_layout()
fig.savefig(FIG_DIR / "topk_jaccard_vs_final.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    stability_population["step"].map(STEP_TO_X),
    stability_population["mean_all_head_spearman_vs_final"],
    marker="o",
)
apply_checkpoint_axis(ax)
ax.set_ylabel("Mean all-head Spearman vs final")
ax.set_ylim(-0.1, 1.05)
ax.set_title("Whole-head RI ranking similarity to the final checkpoint")
fig.tight_layout()
fig.savefig(FIG_DIR / "all_head_spearman_vs_final.png", dpi=180, bbox_inches="tight")
plt.show()



If the mature SIH population were already established during the behavioral transition, we would expect substantial early overlap with the final high-RI heads. Near-zero top-k Jaccard in the early checkpoints instead indicates substantial **head-population turnover/reorganization**. This does not mean that no relational computation exists early; it means the mature top-RI coordinates are not yet a stable set.


# 12. Adjacent Checkpoint Head-Set Churn


In [ ]:

TOP_K = 15
MIN_SUPPORT = 10

adjacent_rows = []

for relation in EXPECTED_RELATIONS:
    top_sets = {}

    for step in STEP_ORDER:
        group = ri[
            (ri["relation"] == relation)
            & (ri["step"] == step)
            & (ri["argmax_scored"] >= MIN_SUPPORT)
        ]
        top = group.nlargest(TOP_K, "ri_argmax")
        top_sets[step] = set(zip(top["layer"], top["head"]))

    for step_a, step_b in zip(STEP_ORDER[:-1], STEP_ORDER[1:]):
        a = top_sets[step_a]
        b = top_sets[step_b]
        union = a | b

        adjacent_rows.append({
            "relation": relation,
            "step_a": step_a,
            "step_b": step_b,
            "jaccard": len(a & b) / len(union) if union else np.nan,
        })

adjacent_stability = pd.DataFrame(adjacent_rows)
adjacent_stability.to_csv(
    RI_ANALYSIS_DIR / "ri_adjacent_topk_stability.csv",
    index=False,
)

adjacent_mean = (
    adjacent_stability.groupby(["step_a", "step_b"])["jaccard"]
    .mean()
    .reset_index()
)

display(adjacent_mean.round(4))


# 13. Identify Early-Moving Heads


In [ ]:

def paired_head_change(step_a, step_b):
    cols = [
        "relation",
        "layer",
        "head",
        "head_name",
        "argmax_scored",
        "qk_argmax_frequency",
        "ri_argmax",
        "ri_per_opportunity",
        "ri_minus_null_random",
    ]

    a = ri[ri["step"] == step_a][cols].copy()
    b = ri[ri["step"] == step_b][cols].copy()

    merged = a.merge(
        b,
        on=["relation", "layer", "head", "head_name"],
        suffixes=(f"_{step_a}", f"_{step_b}"),
    )

    for metric in [
        "qk_argmax_frequency",
        "ri_argmax",
        "ri_per_opportunity",
        "ri_minus_null_random",
    ]:
        merged[f"delta_{metric}"] = (
            merged[f"{metric}_{step_b}"]
            - merged[f"{metric}_{step_a}"]
        )

    merged["conditional_ri_supported_both"] = (
        (merged[f"argmax_scored_{step_a}"] >= MIN_SUPPORT)
        & (merged[f"argmax_scored_{step_b}"] >= MIN_SUPPORT)
    )

    return merged


change_600_900 = paired_head_change(600, 900)
change_600_1000 = paired_head_change(600, 1000)

change_600_900.to_csv(
    RI_ANALYSIS_DIR / "head_changes_step600_to_step900.csv",
    index=False,
)
change_600_1000.to_csv(
    RI_ANALYSIS_DIR / "head_changes_step600_to_step1000.csv",
    index=False,
)

print("Largest increases in RI-per-opportunity, step 600 -> 900:")
display(
    change_600_900.nlargest(
        30,
        "delta_ri_per_opportunity",
    )[
        [
            "relation",
            "head_name",
            "delta_qk_argmax_frequency",
            "delta_ri_argmax",
            "delta_ri_per_opportunity",
            "conditional_ri_supported_both",
        ]
    ].round(5)
)

print("\nLargest supported increases in conditional RI, step 600 -> 1000:")
display(
    change_600_1000[
        change_600_1000["conditional_ri_supported_both"]
    ].nlargest(
        30,
        "delta_ri_argmax",
    )[
        [
            "relation",
            "head_name",
            "delta_qk_argmax_frequency",
            "delta_ri_argmax",
            "delta_ri_per_opportunity",
        ]
    ].round(5)
)



This section is exploratory. The top movers are selected **after observing the developmental change**, so they should not be presented as confirmatory evidence. Their main use is to nominate heads for trajectory inspection or later causal follow-up.

The fixed-final-head analysis above is less selection-flexible and should remain the primary test of whether the mature RI population already existed early.


# 14. Compare RI Development with the Behavioral ICL Sweep


In [ ]:

icl_mean_path = ICL_ANALYSIS_DIR / "mean_20shot_across_tasks.csv"
icl_manifest_path = ICL_RESULTS_DIR / "selected_checkpoint_manifest.csv"

icl_common = pd.DataFrame()

if not icl_mean_path.exists():
    print(
        "ICL analysis file not found:",
        icl_mean_path,
        "\nRI-only analysis is still complete."
    )
else:
    icl_mean = pd.read_csv(icl_mean_path)

    if icl_manifest_path.exists():
        icl_manifest = pd.read_csv(icl_manifest_path)[
            ["step", "tokens_B"]
        ].drop_duplicates()

        icl_mean = icl_mean.merge(
            icl_manifest,
            on="tokens_B",
            how="left",
        )
    else:
        # Exact mapping from the frozen ICL sweep used in this project.
        fallback_step_map = {
            1: 150,
            3: 600,
            5: 1000,
            9: 2000,
            13: 3000,
            17: 4000,
            26: 6000,
            38: 9000,
            80: 19000,
            160: 38000,
            319: 76000,
            638: 152000,
            1498: 357000,
            3896: 928646,
        }
        icl_mean["step"] = icl_mean["tokens_B"].map(fallback_step_map)

    icl_common = population.merge(
        icl_mean[
            [
                "step",
                "tokens_B",
                "mean_accuracy",
                "mean_format_accuracy",
                "mean_label_mass",
            ]
        ],
        on=["step", "tokens_B"],
        how="inner",
    )

    icl_common.to_csv(
        RI_ANALYSIS_DIR / "ri_icl_common_checkpoint_comparison.csv",
        index=False,
    )

    display(
        icl_common[
            [
                "step",
                "tokens_B",
                "mean_accuracy",
                "mean_format_accuracy",
                "mean_qk_argmax_frequency",
                "mean_qk_strict_frequency",
                "mean_ri_argmax",
                "mean_ri_per_opportunity",
                "mean_supported_top15_ri",
            ]
        ].round(5)
    )


In [ ]:

if not icl_common.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(icl_common))
    labels = [
        f"{int(t)}B\nstep {int(s)}"
        for t, s in zip(icl_common["tokens_B"], icl_common["step"])
    ]

    ax.plot(
        x,
        icl_common["mean_accuracy"],
        marker="o",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_xlabel("Common ICL/RI checkpoint")
    ax.set_ylabel("Mean 20-shot accuracy across ICL tasks")
    ax.set_title("Behavioral ICL trajectory at RI checkpoints")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "icl_common_checkpoint_accuracy.png", dpi=180, bbox_inches="tight")
    plt.show()


In [ ]:

if not icl_common.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(icl_common))
    labels = [
        f"{int(t)}B\nstep {int(s)}"
        for t, s in zip(icl_common["tokens_B"], icl_common["step"])
    ]

    ax.plot(
        x,
        icl_common["mean_qk_argmax_frequency"],
        marker="o",
        label="relaxed QK source routing",
    )
    ax.plot(
        x,
        icl_common["mean_qk_strict_frequency"],
        marker="o",
        label="strict QK source routing",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_xlabel("Common ICL/RI checkpoint")
    ax.set_ylabel("Mean routing frequency")
    ax.set_title("RI source-routing trajectory at behavioral checkpoints")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "ri_routing_at_icl_checkpoints.png", dpi=180, bbox_inches="tight")
    plt.show()



### Developmental comparison to make

The behavioral result is already independently established:

- weak pattern discovery at 3B / step 600;
- clear high-shot benefit by 5B / step 1000;
- later task-specific strengthening and eventual behavioral decline.

The RI analysis should therefore ask **which internal statistic changes over that same independently chosen interval**.

Do not move the behavioral boundary after inspecting RI. In particular:

- a QK-routing peak around 4B is temporally aligned with the behavioral transition;
- a later rise in conditional RI is a different phenomenon;
- late high RI at the final checkpoint does not retroactively imply that the mature high-RI heads caused the early ICL transition.


# 15. Optional Cross-Reference with the Previous Single-Hop Head Inventory


In [ ]:

# These are the 25 |I_h| >= 0.3 heads from the completed Stage-3
# single-hop characterization. This section only asks about coordinate
# overlap; it does NOT equate AGENDA RI with causal importance.

PREVIOUS_STRONG_HEADS = {
    "L17H1", "L18H19", "L27H6", "L18H18", "L21H18",
    "L25H17", "L22H5", "L16H21", "L16H1", "L21H6",
    "L15H25", "L30H13", "L21H23", "L17H17", "L17H24",
    "L19H22", "L14H26", "L26H23", "L18H24", "L17H3",
    "L13H10", "L19H16", "L23H15", "L30H18", "L20H1",
}

overlap = (
    final_top_heads[
        final_top_heads["head_name"].isin(PREVIOUS_STRONG_HEADS)
    ]
    .sort_values(["relation", "rank"])
    .reset_index(drop=True)
)

overlap.to_csv(
    RI_ANALYSIS_DIR / "final_ri_top15_overlap_previous_singlehop_heads.csv",
    index=False,
)

print(
    "Unique previous strong heads appearing in at least one final AGENDA top-15:",
    overlap["head_name"].nunique(),
    "/",
    len(PREVIOUS_STRONG_HEADS),
)
display(overlap)


In [ ]:

if not overlap.empty:
    overlap_names = sorted(overlap["head_name"].unique())

    overlap_trace = ri[
        ri["head_name"].isin(overlap_names)
    ][
        [
            "step",
            "tokens_B",
            "relation",
            "head_name",
            "argmax_scored",
            "qk_argmax_frequency",
            "ri_argmax",
            "ri_per_opportunity",
        ]
    ].copy()

    overlap_trace.to_csv(
        RI_ANALYSIS_DIR / "previous_singlehop_overlap_head_trajectories.csv",
        index=False,
    )

    print("Overlapping head coordinates:")
    print(", ".join(overlap_names))



Coordinate overlap is interesting but must be interpreted carefully. The previous experiments showed that RI was a weak causal ranking for the synthetic mother-of task. Therefore, finding a previously causal head among AGENDA's mature high-RI heads does **not** validate RI as a causal selector; conversely, lack of overlap would not invalidate the developmental timing analysis.


# 16. Programmatic Landmark Summary


In [ ]:

qk_peak = population.loc[
    population["mean_qk_argmax_frequency"].idxmax()
]
strict_peak = population.loc[
    population["mean_qk_strict_frequency"].idxmax()
]
peropp_peak = population.loc[
    population["mean_ri_per_opportunity"].idxmax()
]

early = population[population["step"].isin(EARLY_STEPS)]
cond_range = (
    early["mean_ri_argmax"].min(),
    early["mean_ri_argmax"].max(),
)

final_stability_pre5 = stability_population[
    stability_population["step"] <= 1000
]

print("Developmental landmarks")
print("-----------------------")
print(
    f"Relaxed QK routing peak: {qk_peak.tokens_B:.0f}B, "
    f"step {qk_peak.step:.0f}, mean={qk_peak.mean_qk_argmax_frequency:.5f}"
)
print(
    f"Strict QK routing peak: {strict_peak.tokens_B:.0f}B, "
    f"step {strict_peak.step:.0f}, mean={strict_peak.mean_qk_strict_frequency:.5f}"
)
print(
    f"RI-per-opportunity peak: {peropp_peak.tokens_B:.0f}B, "
    f"step {peropp_peak.step:.0f}, mean={peropp_peak.mean_ri_per_opportunity:.6f}"
)
print(
    "Mean conditional RI across the 3B-5B dense window ranges only from "
    f"{cond_range[0]:.5f} to {cond_range[1]:.5f}."
)
print(
    "Largest mean top-k Jaccard with FINAL heads through step 1000: "
    f"{final_stability_pre5['mean_topk_jaccard_vs_final'].max():.4f}"
)

if not icl_common.empty:
    row3 = icl_common[icl_common["step"] == 600]
    row5 = icl_common[icl_common["step"] == 1000]

    if len(row3) == 1 and len(row5) == 1:
        print(
            "Mean 20-shot ICL accuracy: "
            f"{row3.iloc[0].mean_accuracy:.3f} at 3B/600 -> "
            f"{row5.iloc[0].mean_accuracy:.3f} at 5B/1000."
        )



# 17. Interpretation Checklist

The final written conclusion should answer these separately:

1. **Behavioral timing:** when does ICL pattern discovery change?
2. **Routing timing:** when does source-directed QK attention change?
3. **Conditional semantic selectivity:** when does `ri_argmax` itself change?
4. **Combined signal:** when does RI-per-opportunity change?
5. **Head identity:** are the same high-RI coordinates stable through training?
6. **Null sensitivity:** are target scores distinguishable from the saved null controls?
7. **Strict-gate sensitivity:** does the early pattern survive `tau = 2.2`?
8. **Prior-project consistency:** do any mature AGENDA high-RI heads overlap previous single-hop causal heads, and does that alter the earlier conclusion that RI is not a reliable causal ranking?

### Current-run intermediate conclusion

The uploaded sweep currently points to a nuanced result:

> **The clearest RI-side event near the 3B–5B behavioral ICL transition is an increase in source-directed attention routing, peaking around the 4B checkpoints. Population-average conditional RI does not show an equally sharp early jump. Mature high-RI head identity is also highly unstable relative to the final model.**

This is partial developmental alignment, not a full replication of a claim that mature Semantic Induction Heads themselves appear wholesale at the ICL boundary. The distinction is scientifically useful and should be preserved in the report.


# 18. Saved Analysis Outputs


In [ ]:

print("Analysis directory:", RI_ANALYSIS_DIR)
print("\nDerived CSVs:")
for path in sorted(RI_ANALYSIS_DIR.glob("*.csv")):
    print(" -", path.name)

print("\nFigures:")
for path in sorted(FIG_DIR.glob("*.png")):
    print(" -", path.name)


FURTHER analysis at the 3b-5b transition

In [ ]:
# ============================================================
# Paired cluster bootstrap for developmental RI changes
# ============================================================

BOOTSTRAP_PAIRS = [
    (600, 850),
    (600, 900),
    (600, 1000),
]

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260924

# These are the additive sufficient statistics needed to
# reconstruct QK routing and conditional RI.
BOOTSTRAP_FIELDS = [
    "opportunities",
    "argmax_hits",
    "argmax_scored",
    "ri_sum",
]


def revision_for_step(step):
    rows = checkpoint_manifest.loc[
        checkpoint_manifest["step"] == step
    ]

    if len(rows) != 1:
        raise RuntimeError(
            f"Expected exactly one checkpoint for step {step}, "
            f"found {len(rows)}."
        )

    return rows.iloc[0]["revision"]


def chunk_paths_for_step(step):
    revision = revision_for_step(step)

    chunk_dir = (
        RI_RUN_DIR
        / "chunks"
        / revision
    )

    paths = sorted(chunk_dir.glob("chunk_*.npz"))

    if not paths:
        raise FileNotFoundError(
            f"No RI chunks found for step {step}: {chunk_dir}"
        )

    return paths


def load_bootstrap_chunk_stack(step):
    """
    Returns one array per accumulator field.

    Shape of every returned field:
        [n_chunks, n_relations, n_directions, n_layers, n_heads]
    """

    paths = chunk_paths_for_step(step)

    per_field = {
        field: []
        for field in BOOTSTRAP_FIELDS
    }

    for path in paths:
        with np.load(path) as data:

            for field in BOOTSTRAP_FIELDS:
                if field not in data:
                    raise KeyError(
                        f"{field} missing from {path}"
                    )

                per_field[field].append(
                    np.asarray(data[field])
                )

    stacked = {
        field: np.stack(arrays, axis=0)
        for field, arrays in per_field.items()
    }

    return {
        "step": step,
        "revision": revision_for_step(step),
        "paths": paths,
        "arrays": stacked,
    }

In [ ]:
# preload the four checkpoints:
BOOTSTRAP_STEPS = sorted({
    step
    for pair in BOOTSTRAP_PAIRS
    for step in pair
})

bootstrap_data = {
    step: load_bootstrap_chunk_stack(step)
    for step in BOOTSTRAP_STEPS
}


# Verify that exactly the same chunk indices exist at every checkpoint.
reference_names = [
    path.name
    for path in bootstrap_data[BOOTSTRAP_STEPS[0]]["paths"]
]

for step in BOOTSTRAP_STEPS[1:]:
    names = [
        path.name
        for path in bootstrap_data[step]["paths"]
    ]

    assert names == reference_names, (
        f"Chunk structure differs at step {step}. "
        "Paired bootstrap requires identical chunking."
    )


N_CHUNKS = len(reference_names)

print("Bootstrap checkpoints:", BOOTSTRAP_STEPS)
print("Paired resampling units:", N_CHUNKS, "chunks")
print(
    "Text groups per chunk:",
    run_config["text_groups_per_chunk"],
)

In [ ]:
# Reconstruct the statistics from an arbitrary bootstrap sample
def safe_ratio(num, den):
    num = np.asarray(num, dtype=np.float64)
    den = np.asarray(den, dtype=np.float64)

    out = np.full_like(num, np.nan, dtype=np.float64)

    np.divide(
        num,
        den,
        out=out,
        where=(den > 0),
    )

    return out


def weighted_accumulator(chunk_data, weights):
    """
    Recombine chunk-level sufficient statistics according to
    bootstrap multiplicities.

    weights[c] = number of times chunk c is present in the
    bootstrap sample.
    """

    result = {}

    for field in BOOTSTRAP_FIELDS:

        # chunk_data[field]:
        # [chunk, relation, direction, layer, head]
        result[field] = np.einsum(
            "c,crdlh->rdlh",
            weights,
            chunk_data[field],
            optimize=True,
        )

    return result


def summarize_bootstrap_accumulator(acc):
    """
    Produce relation-specific and relation-balanced statistics.

    Heads are NOT treated as bootstrap units. They remain fixed
    model components.

    Returns:
        relation_qk: [R]
        relation_ri: [R]
        overall_qk: scalar
        overall_ri: scalar
    """

    # Only forward direction exists in this sweep.
    direction_index = 0

    qk = safe_ratio(
        acc["argmax_hits"],
        acc["opportunities"],
    )[:, direction_index]

    conditional_ri = safe_ratio(
        acc["ri_sum"],
        acc["argmax_scored"],
    )[:, direction_index]

    # [relation, layer, head] -> [relation]
    relation_qk = np.nanmean(
        qk,
        axis=(1, 2),
    )

    relation_ri = np.nanmean(
        conditional_ri,
        axis=(1, 2),
    )

    # Equal weight for each of the seven relations.
    overall_qk = np.nanmean(relation_qk)
    overall_ri = np.nanmean(relation_ri)

    return {
        "relation_qk": relation_qk,
        "relation_ri": relation_ri,
        "overall_qk": overall_qk,
        "overall_ri": overall_ri,
    }

In [ ]:
#verify that the chunk reconstruction reproduces your existing results

def full_sample_summary(step):
    arrays = bootstrap_data[step]["arrays"]

    weights = np.ones(
        len(bootstrap_data[step]["paths"]),
        dtype=np.int64,
    )

    acc = weighted_accumulator(
        arrays,
        weights,
    )

    return summarize_bootstrap_accumulator(acc)


for step in BOOTSTRAP_STEPS:

    summary = full_sample_summary(step)

    print(f"\nSTEP {step}")

    for relation, qk, ri_value in zip(
        EXPECTED_RELATIONS,
        summary["relation_qk"],
        summary["relation_ri"],
    ):
        print(
            f"{relation:14s} "
            f"QK={qk:.6f}  "
            f"RI={ri_value:.6f}"
        )

    print(
        "Relation-balanced overall:",
        f"QK={summary['overall_qk']:.6f}",
        f"RI={summary['overall_ri']:.6f}",
    )

In [ ]:
# The actual paired bootstrap

rng = np.random.default_rng(BOOTSTRAP_SEED)

bootstrap_records = []


for step_a, step_b in BOOTSTRAP_PAIRS:

    data_a = bootstrap_data[step_a]["arrays"]
    data_b = bootstrap_data[step_b]["arrays"]

    n_chunks_a = data_a["opportunities"].shape[0]
    n_chunks_b = data_b["opportunities"].shape[0]

    assert n_chunks_a == n_chunks_b

    n_chunks = n_chunks_a

    # --------------------------------------------------------
    # Observed difference on the full dataset
    # --------------------------------------------------------

    full_weights = np.ones(
        n_chunks,
        dtype=np.int64,
    )

    observed_a = summarize_bootstrap_accumulator(
        weighted_accumulator(
            data_a,
            full_weights,
        )
    )

    observed_b = summarize_bootstrap_accumulator(
        weighted_accumulator(
            data_b,
            full_weights,
        )
    )

    observed_qk_delta = (
        observed_b["relation_qk"]
        - observed_a["relation_qk"]
    )

    observed_ri_delta = (
        observed_b["relation_ri"]
        - observed_a["relation_ri"]
    )

    observed_overall_qk_delta = (
        observed_b["overall_qk"]
        - observed_a["overall_qk"]
    )

    observed_overall_ri_delta = (
        observed_b["overall_ri"]
        - observed_a["overall_ri"]
    )

    # --------------------------------------------------------
    # Bootstrap distributions
    # --------------------------------------------------------

    boot_qk = np.empty(
        (N_BOOTSTRAP, len(EXPECTED_RELATIONS)),
        dtype=np.float64,
    )

    boot_ri = np.empty_like(boot_qk)

    boot_overall_qk = np.empty(
        N_BOOTSTRAP,
        dtype=np.float64,
    )

    boot_overall_ri = np.empty(
        N_BOOTSTRAP,
        dtype=np.float64,
    )

    for b in range(N_BOOTSTRAP):

        # SAME sampled chunk multiplicities for both checkpoints.
        #
        # This is what makes the bootstrap paired.
        weights = rng.multinomial(
            n_chunks,
            np.full(n_chunks, 1.0 / n_chunks),
        )

        summary_a = summarize_bootstrap_accumulator(
            weighted_accumulator(
                data_a,
                weights,
            )
        )

        summary_b = summarize_bootstrap_accumulator(
            weighted_accumulator(
                data_b,
                weights,
            )
        )

        boot_qk[b] = (
            summary_b["relation_qk"]
            - summary_a["relation_qk"]
        )

        boot_ri[b] = (
            summary_b["relation_ri"]
            - summary_a["relation_ri"]
        )

        boot_overall_qk[b] = (
            summary_b["overall_qk"]
            - summary_a["overall_qk"]
        )

        boot_overall_ri[b] = (
            summary_b["overall_ri"]
            - summary_a["overall_ri"]
        )

    # --------------------------------------------------------
    # Relation-specific confidence intervals
    # --------------------------------------------------------

    for r, relation in enumerate(EXPECTED_RELATIONS):

        for metric, observed, samples in [
            (
                "qk_argmax_frequency",
                observed_qk_delta[r],
                boot_qk[:, r],
            ),
            (
                "conditional_ri",
                observed_ri_delta[r],
                boot_ri[:, r],
            ),
        ]:

            ci_low, ci_high = np.nanquantile(
                samples,
                [0.025, 0.975],
            )

            bootstrap_records.append({
                "step_from": step_a,
                "step_to": step_b,
                "scope": relation,
                "metric": metric,
                "observed_delta": observed,
                "ci_2_5": ci_low,
                "ci_97_5": ci_high,
                "fraction_bootstrap_positive": (
                    np.nanmean(samples > 0)
                ),
                "n_bootstrap": N_BOOTSTRAP,
                "resampling_unit": "20-text-group chunk",
            })

    # --------------------------------------------------------
    # Relation-balanced overall confidence intervals
    # --------------------------------------------------------

    for metric, observed, samples in [
        (
            "qk_argmax_frequency",
            observed_overall_qk_delta,
            boot_overall_qk,
        ),
        (
            "conditional_ri",
            observed_overall_ri_delta,
            boot_overall_ri,
        ),
    ]:

        ci_low, ci_high = np.nanquantile(
            samples,
            [0.025, 0.975],
        )

        bootstrap_records.append({
            "step_from": step_a,
            "step_to": step_b,
            "scope": "RELATION_BALANCED_OVERALL",
            "metric": metric,
            "observed_delta": observed,
            "ci_2_5": ci_low,
            "ci_97_5": ci_high,
            "fraction_bootstrap_positive": (
                np.nanmean(samples > 0)
            ),
            "n_bootstrap": N_BOOTSTRAP,
            "resampling_unit": "20-text-group chunk",
        })


paired_bootstrap_ci = pd.DataFrame(
    bootstrap_records
)

paired_bootstrap_ci.to_csv(
    RI_ANALYSIS_DIR
    / "ri_paired_chunk_bootstrap_ci.csv",
    index=False,
)

print(
    paired_bootstrap_ci.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

Last check - Fixed support anaysis

In [ ]:
# build fixed support mask

# ============================================================
# Fixed-support sensitivity analysis for conditional RI
# ============================================================

FIXED_SUPPORT_MIN_SCORED = 10

FIXED_SUPPORT_PAIRS = [
    (600, 850),
    (600, 900),
    (600, 1000),
]


def full_accumulator(step):
    """
    Combine every saved chunk once to reconstruct the full-data
    accumulator for one checkpoint.
    """
    arrays = bootstrap_data[step]["arrays"]

    n_chunks = arrays["argmax_scored"].shape[0]

    weights = np.ones(
        n_chunks,
        dtype=np.int64,
    )

    return weighted_accumulator(
        arrays,
        weights,
    )


def make_fixed_support_mask(step_a, step_b, min_scored=10):
    """
    A head is included only when it has at least min_scored
    conditional-RI observations in the FULL dataset at BOTH
    endpoints.

    Shape returned:
        [relation, layer, head]

    The RI sweep contains forward direction only, hence [:, 0].
    """

    acc_a = full_accumulator(step_a)
    acc_b = full_accumulator(step_b)

    scored_a = acc_a["argmax_scored"][:, 0]
    scored_b = acc_b["argmax_scored"][:, 0]

    mask = (
        (scored_a >= min_scored)
        & (scored_b >= min_scored)
    )

    return mask


fixed_support_masks = {}

for step_a, step_b in FIXED_SUPPORT_PAIRS:

    mask = make_fixed_support_mask(
        step_a,
        step_b,
        min_scored=FIXED_SUPPORT_MIN_SCORED,
    )

    fixed_support_masks[(step_a, step_b)] = mask

    print(f"\nFixed support: {step_a} -> {step_b}")

    for r, relation in enumerate(EXPECTED_RELATIONS):
        print(
            f"{relation:14s}: "
            f"{int(mask[r].sum())} heads"
        )

In [ ]:
# define fixed support RI summary

def summarize_fixed_support_ri(acc, fixed_mask):
    """
    Compute conditional RI using a head population fixed BEFORE
    bootstrap resampling.

    acc:
        additive accumulator after combining bootstrap chunks

    fixed_mask:
        [relation, layer, head]

    Returns:
        relation_ri: one value for each relation
        overall_ri: equal-weight average across relations
        zero_denominator_heads: count of fixed heads whose bootstrap
                                replicate contains no scored events
    """

    # Forward direction only.
    ri_sum = acc["ri_sum"][:, 0]
    argmax_scored = acc["argmax_scored"][:, 0]

    conditional_ri = safe_ratio(
        ri_sum,
        argmax_scored,
    )

    relation_values = []
    zero_denominator_heads = 0

    for r in range(len(EXPECTED_RELATIONS)):

        selected = fixed_mask[r]

        values = conditional_ri[r][selected]

        zero_denominator_heads += int(
            np.isnan(values).sum()
        )

        relation_values.append(
            np.nanmean(values)
        )

    relation_ri = np.asarray(
        relation_values,
        dtype=np.float64,
    )

    # Equal relation weighting, exactly as in the previous
    # relation-balanced bootstrap.
    overall_ri = np.nanmean(relation_ri)

    return {
        "relation_ri": relation_ri,
        "overall_ri": overall_ri,
        "zero_denominator_heads": zero_denominator_heads,
    }

In [ ]:
# ============================================================
# Fixed-support paired BAYESIAN cluster bootstrap
#
# Why Bayesian bootstrap here?
# Ordinary resampling-with-replacement can omit all chunks
# containing the scored events of a low-support head.
#
# Dirichlet weights are strictly positive, so every fixed head
# remains defined in every replicate.
# ============================================================

N_FIXED_BOOTSTRAP = 5000
FIXED_BOOTSTRAP_SEED = 20260924

rng_fixed = np.random.default_rng(
    FIXED_BOOTSTRAP_SEED
)

fixed_support_bayes_records = []


for step_a, step_b in FIXED_SUPPORT_PAIRS:

    print(
        f"\nRunning Bayesian fixed-support bootstrap: "
        f"{step_a} -> {step_b}"
    )

    data_a = bootstrap_data[step_a]["arrays"]
    data_b = bootstrap_data[step_b]["arrays"]

    fixed_mask = fixed_support_masks[
        (step_a, step_b)
    ]

    n_chunks = data_a["argmax_scored"].shape[0]

    assert (
        data_b["argmax_scored"].shape[0]
        == n_chunks
    )

    # --------------------------------------------------------
    # Observed full-data statistic
    # --------------------------------------------------------

    full_weights = np.ones(
        n_chunks,
        dtype=np.float64,
    )

    full_a = weighted_accumulator(
        data_a,
        full_weights,
    )

    full_b = weighted_accumulator(
        data_b,
        full_weights,
    )

    obs_a = summarize_fixed_support_ri(
        full_a,
        fixed_mask,
    )

    obs_b = summarize_fixed_support_ri(
        full_b,
        fixed_mask,
    )

    observed_relation_delta = (
        obs_b["relation_ri"]
        - obs_a["relation_ri"]
    )

    observed_overall_delta = (
        obs_b["overall_ri"]
        - obs_a["overall_ri"]
    )

    # --------------------------------------------------------
    # Bayesian bootstrap distributions
    # --------------------------------------------------------

    boot_relation_delta = np.empty(
        (
            N_FIXED_BOOTSTRAP,
            len(EXPECTED_RELATIONS),
        ),
        dtype=np.float64,
    )

    boot_overall_delta = np.empty(
        N_FIXED_BOOTSTRAP,
        dtype=np.float64,
    )

    zero_denom_counts = np.zeros(
        N_FIXED_BOOTSTRAP,
        dtype=np.int64,
    )

    for b in range(N_FIXED_BOOTSTRAP):

        # Strictly positive weights.
        #
        # SAME weights are used at both checkpoints,
        # preserving the paired design.
        weights = rng_fixed.dirichlet(
            np.ones(n_chunks)
        )

        acc_a = weighted_accumulator(
            data_a,
            weights,
        )

        acc_b = weighted_accumulator(
            data_b,
            weights,
        )

        summary_a = summarize_fixed_support_ri(
            acc_a,
            fixed_mask,
        )

        summary_b = summarize_fixed_support_ri(
            acc_b,
            fixed_mask,
        )

        boot_relation_delta[b] = (
            summary_b["relation_ri"]
            - summary_a["relation_ri"]
        )

        boot_overall_delta[b] = (
            summary_b["overall_ri"]
            - summary_a["overall_ri"]
        )

        zero_denom_counts[b] = (
            summary_a["zero_denominator_heads"]
            + summary_b["zero_denominator_heads"]
        )

    # This should now be exactly zero.
    print(
        "Replicates with any zero-denominator fixed head:",
        np.mean(zero_denom_counts > 0),
    )

    # --------------------------------------------------------
    # Relation-specific results
    # --------------------------------------------------------

    for r, relation in enumerate(EXPECTED_RELATIONS):

        samples = boot_relation_delta[:, r]

        ci_low, ci_high = np.quantile(
            samples,
            [0.025, 0.975],
        )

        fixed_support_bayes_records.append({
            "step_from": step_a,
            "step_to": step_b,
            "scope": relation,
            "metric":
                "conditional_ri_fixed_support",
            "bootstrap_type":
                "paired_bayesian_cluster",
            "n_fixed_heads":
                int(fixed_mask[r].sum()),
            "observed_delta":
                observed_relation_delta[r],
            "ci_2_5": ci_low,
            "ci_97_5": ci_high,
            "fraction_bootstrap_positive":
                np.mean(samples > 0),
            "n_bootstrap":
                N_FIXED_BOOTSTRAP,
            "min_scored_both_endpoints":
                FIXED_SUPPORT_MIN_SCORED,
            "resampling_unit":
                "20-text-group chunk",
        })

    # --------------------------------------------------------
    # Relation-balanced overall result
    # --------------------------------------------------------

    ci_low, ci_high = np.quantile(
        boot_overall_delta,
        [0.025, 0.975],
    )

    fixed_support_bayes_records.append({
        "step_from": step_a,
        "step_to": step_b,
        "scope":
            "RELATION_BALANCED_OVERALL",
        "metric":
            "conditional_ri_fixed_support",
        "bootstrap_type":
            "paired_bayesian_cluster",
        "n_fixed_heads":
            int(fixed_mask.sum()),
        "observed_delta":
            observed_overall_delta,
        "ci_2_5": ci_low,
        "ci_97_5": ci_high,
        "fraction_bootstrap_positive":
            np.mean(boot_overall_delta > 0),
        "n_bootstrap":
            N_FIXED_BOOTSTRAP,
        "min_scored_both_endpoints":
            FIXED_SUPPORT_MIN_SCORED,
        "resampling_unit":
            "20-text-group chunk",
    })


fixed_support_bayes_ci = pd.DataFrame(
    fixed_support_bayes_records
)

fixed_support_bayes_ci.to_csv(
    RI_ANALYSIS_DIR
    / "ri_fixed_support_bayesian_bootstrap_ci.csv",
    index=False,
)

print("\nRelation-balanced fixed-support results:\n")

print(
    fixed_support_bayes_ci[
        fixed_support_bayes_ci["scope"]
        == "RELATION_BALANCED_OVERALL"
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)